## Working with Redis from Python

In [1]:
import sys
!{sys.executable} -m pip install redis

In [1]:
import redis

# Choose server 'redis-1' in docker network or 'localhost' from outside
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
#r.auth('abc123!')
r.ping()

True

In [9]:
# Store individual movie attributes
r.set('movie:0110912:title', 'Pulp Fiction')
r.set('movie:0110912:year',  '1994')
r.set('movie:0110912:rating','8.9')

print(r.get('movie:0110912:title'))    # Pulp Fiction
print(r.exists('movie:0110912:title')) # 1

# Set multiple titles at once
r.mset({
    'movie:0111161:title': 'The Shawshank Redemption',
    'movie:0068646:title': 'The Godfather',
    'movie:0468569:title': 'The Dark Knight',
})
print(r.mget('movie:0111161:title', 'movie:0068646:title', 'movie:0468569:title'))
# ['The Shawshank Redemption', 'The Godfather', 'The Dark Knight']

# SETNX — only sets the key if it does not already exist
print(r.setnx('movie:0110912:title', 'Something Else'))  # False — key exists
print(r.get('movie:0110912:title'))                       # Pulp Fiction

# Featured movie with a TTL
r.set('featured:movie', 'The Matrix', ex=120)
print(r.ttl('featured:movie'))         # ~120

Pulp Fiction
1
['The Shawshank Redemption', 'The Godfather', 'The Dark Knight']
False
Pulp Fiction
120


In [10]:
# Initialise and increment a movie view counter
r.set('movie:0110912:views', 0)
print(r.incr('movie:0110912:views'))           # 1
print(r.incrby('movie:0110912:views', 1000))   # 1001
print(r.decr('movie:0110912:views'))           # 1000
print(r.decrby('movie:0110912:views', 500))    # 500

# INCR on a non-existing key starts from 0
r.delete('movie:0133093:views')
print(r.incr('movie:0133093:views'))           # 1

1
1001
1000
500
1


In [11]:
r.delete('watchlist')

# Build a watchlist
r.rpush('watchlist', 'Pulp Fiction')
r.rpush('watchlist', 'The Matrix')
r.rpush('watchlist', 'Inception')
r.lpush('watchlist', 'The Dark Knight')   # jump to the front of the queue

print(r.lrange('watchlist', 0, -1))
# ['The Dark Knight', 'Pulp Fiction', 'The Matrix', 'Inception']

print(r.llen('watchlist'))                # 4

# Watch the next movie (remove from the front)
print(r.lpop('watchlist'))                # The Dark Knight

# Drop the last movie from the queue
print(r.rpop('watchlist'))                # Inception

print(r.lrange('watchlist', 0, -1))
# ['Pulp Fiction', 'The Matrix']

['The Dark Knight', 'Pulp Fiction', 'The Matrix', 'Inception']
4
The Dark Knight
Inception
['Pulp Fiction', 'The Matrix']


In [12]:
r.delete('genre:action', 'genre:sci-fi', 'genre:action-or-sci-fi')

r.sadd('genre:action', 'The Dark Knight', 'Avengers: Endgame', 'The Matrix', 'Inception')
r.sadd('genre:sci-fi', 'The Matrix', 'Inception', 'Interstellar', 'Avengers: Endgame')

print(r.smembers('genre:action'))
# {'The Dark Knight', 'Avengers: Endgame', 'The Matrix', 'Inception'}  (order may vary)

# Membership test
print(r.sismember('genre:action', 'The Dark Knight'))  # True
print(r.sismember('genre:action', 'Pulp Fiction'))     # False

# Remove a movie from a genre
r.srem('genre:action', 'Avengers: Endgame')

# SUNION — all movies tagged as Action OR Sci-Fi
print(r.sunion('genre:action', 'genre:sci-fi'))
# {'The Dark Knight', 'The Matrix', 'Inception', 'Interstellar', 'Avengers: Endgame'}

# SINTER — movies tagged as both Action AND Sci-Fi
print(r.sinter('genre:action', 'genre:sci-fi'))
# {'The Matrix', 'Inception'}

# SDIFF — movies tagged as Action but NOT Sci-Fi
print(r.sdiff('genre:action', 'genre:sci-fi'))
# {'The Dark Knight'}r.delete('top:movies')

ratings = {
    'The Shawshank Redemption': 9.2,
    'The Godfather':            9.2,
    "The Godfather: Part II":   9.0,
    'The Dark Knight':          9.0,
    '12 Angry Men':             8.9,
    'Pulp Fiction':             8.9,
    "Schindler's List":         8.9,
    'The Matrix':               8.7,
    'Inception':                8.7,
    'Forrest Gump':             8.7,
}
r.zadd('top:movies', ratings)

# ZREVRANGE — top 5 highest-rated (descending), with scores
print(r.zrevrange('top:movies', 0, 4, withscores=True))
# [('The Shawshank Redemption', 9.2), ('The Godfather', 9.2),
#  ('The Dark Knight', 9.0), ("The Godfather: Part II", 9.0), ("Schindler's List", 8.9)]

# ZRANGE — lowest-rated first (ascending), first 3
print(r.zrange('top:movies', 0, 2))
# ['Forrest Gump', 'Inception', 'The Matrix']

# ZSCORE — look up a specific movie's rating
print(r.zscore('top:movies', 'Pulp Fiction'))      # 8.9

# ZREVRANK — position in the leaderboard (0 = highest rated)
print(r.zrevrank('top:movies', 'The Shawshank Redemption'))  # 0
print(r.zrevrank('top:movies', 'The Matrix'))                 # 7

{'Avengers: Endgame', 'The Dark Knight', 'The Matrix', 'Inception'}
1
0
{'Inception', 'Interstellar', 'The Dark Knight', 'The Matrix', 'Avengers: Endgame'}
{'The Matrix', 'Inception'}
{'The Dark Knight'}
[('The Shawshank Redemption', 9.2), ('The Godfather', 9.2), ('The Godfather: Part II', 9.0), ('The Dark Knight', 9.0), ("Schindler's List", 8.9)]
['Forrest Gump', 'Inception', 'The Matrix']
8.9
0
7


In [13]:
r.delete('movie:0110912', 'movie:0133093', 'movie:0111161')

# Store a full movie object as a hash
r.hset('movie:0110912', mapping={
    'title':   'Pulp Fiction',
    'year':    '1994',
    'runtime': '154',
    'rating':  '8.9',
    'votes':   '2084331',
})

print(r.hgetall('movie:0110912'))
# {'title': 'Pulp Fiction', 'year': '1994', 'runtime': '154', 'rating': '8.9', 'votes': '2084331'}

r.hset('movie:0133093', mapping={
    'title':   'The Matrix',
    'year':    '1999',
    'runtime': '136',
    'rating':  '8.7',
    'votes':   '1496538',
})

# HGET — retrieve a single field
print(r.hget('movie:0110912', 'title'))    # Pulp Fiction
print(r.hget('movie:0110912', 'rating'))   # 8.9

# HINCRBY — atomically increment the vote count
print(r.hincrby('movie:0110912', 'votes', 1))       # 2084332
print(r.hincrby('movie:0110912', 'votes', 1000))    # 2085332

# HDEL — remove a field
r.hdel('movie:0110912', 'votes')
print(r.hgetall('movie:0110912'))
# {'title': 'Pulp Fiction', 'year': '1994', 'runtime': '154', 'rating': '8.9'}

{'title': 'Pulp Fiction', 'year': '1994', 'runtime': '154', 'rating': '8.9', 'votes': '2084331'}
Pulp Fiction
8.9
2084332
2085332
{'title': 'Pulp Fiction', 'year': '1994', 'runtime': '154', 'rating': '8.9'}


In [14]:
top_movies = [
    ('0111161', 'The Shawshank Redemption',                              1994, 9.2),
    ('0068646', 'The Godfather',                                         1972, 9.2),
    ('0071562', 'The Godfather: Part II',                                1974, 9.0),
    ('0468569', 'The Dark Knight',                                       2008, 9.0),
    ('0050083', '12 Angry Men',                                          1957, 8.9),
    ('0110912', 'Pulp Fiction',                                          1994, 8.9),
    ('0108052', "Schindler's List",                                      1993, 8.9),
    ('0167260', 'The Lord of the Rings: The Return of the King',         2003, 8.9),
    ('0060196', 'The Good, the Bad and the Ugly',                        1966, 8.8),
    ('0137523', 'Fight Club',                                            1999, 8.8),
    ('4154796', 'Avengers: Endgame',                                     2019, 8.8),
    ('0120737', 'The Lord of the Rings: The Fellowship of the Ring',     2001, 8.8),
    ('0109830', 'Forrest Gump',                                          1994, 8.7),
    ('0080684', 'Star Wars: Episode V - The Empire Strikes Back',        1980, 8.7),
    ('1375666', 'Inception',                                             2010, 8.7),
    ('0167261', 'The Lord of the Rings: The Two Towers',                 2002, 8.7),
    ('0073486', "One Flew Over the Cuckoo's Nest",                       1975, 8.7),
    ('0099685', 'Goodfellas',                                            1990, 8.7),
    ('0133093', 'The Matrix',                                            1999, 8.7),
    ('0047478', 'Seven Samurai',                                         1954, 8.6),
    ('0114369', 'Se7en',                                                 1995, 8.6),
    ('0317248', 'City of God',                                           2002, 8.6),
    ('0076759', 'Star Wars: Episode IV - A New Hope',                    1977, 8.6),
    ('0102926', 'The Silence of the Lambs',                              1991, 8.6),
    ('0038650', "It's a Wonderful Life",                                 1946, 8.6),
    ('0118799', 'Life Is Beautiful',                                     1997, 8.6),
    ('0245429', 'Spirited Away',                                         2001, 8.5),
    ('0120815', 'Saving Private Ryan',                                   1998, 8.5),
    ('0114814', 'The Usual Suspects',                                    1995, 8.5),
    ('0110413', 'Léon: The Professional',                                1994, 8.5),
    ('0120689', 'The Green Mile',                                        1999, 8.5),
    ('0816692', 'Interstellar',                                          2014, 8.5),
    ('0054215', 'Psycho',                                                1960, 8.5),
    ('0120586', 'American History X',                                    1998, 8.5),
    ('0021749', 'City Lights',                                           1931, 8.5),
    ('0034583', 'Casablanca',                                            1942, 8.5),
    ('0064116', 'Once Upon a Time in the West',                          1968, 8.5),
    ('0253474', 'The Pianist',                                           2002, 8.5),
    ('0027977', 'Modern Times',                                          1936, 8.5),
    ('1675434', 'The Intouchables',                                      2011, 8.5),
    ('0407887', 'The Departed',                                          2006, 8.5),
    ('0088763', 'Back to the Future',                                    1985, 8.5),
    ('0103064', 'Terminator 2: Judgment Day',                            1991, 8.5),
    ('2582802', 'Whiplash',                                              2014, 8.5),
    ('0110357', 'The Lion King',                                         1994, 8.5),
    ('0047396', 'Rear Window',                                           1954, 8.5),
    ('0082971', 'Raiders of the Lost Ark',                               1981, 8.5),
    ('0172495', 'Gladiator',                                             2000, 8.5),
    ('0482571', 'The Prestige',                                          2006, 8.5),
    ('0078788', 'Apocalypse Now',                                        1979, 8.4),
]

pipe = r.pipeline()

for movie_id, title, year, rating in top_movies:
    key = f'movie:{movie_id}'
    pipe.hset(key, mapping={'title': title, 'year': str(year), 'rating': str(rating)})
    pipe.zadd('top50:movies', {title: rating})

pipe.execute()   # all commands sent in one round trip

print(f'Loaded {len(top_movies)} movies.')

# Verify: top 5 by rating
print(r.zrevrange('top50:movies', 0, 4, withscores=True))

Loaded 50 movies.
[('The Shawshank Redemption', 9.2), ('The Godfather', 9.2), ('The Godfather: Part II', 9.0), ('The Dark Knight', 9.0), ('The Lord of the Rings: The Return of the King', 8.9)]


In [8]:
pattern_groups = [
    'movie:*',
    'top:movies',
    'top50:movies',
    'genre:*',
    'watchlist',
    'featured:movie',
]

keys_to_delete = []
for pattern in pattern_groups:
    keys_to_delete.extend(r.keys(pattern))

if keys_to_delete:
    r.delete(*keys_to_delete)
    print(f'Deleted {len(keys_to_delete)} keys.')
else:
    print('Nothing to delete.')

Deleted 64 keys.
